In [4]:
#!/usr/bin/env python
# coding: utf-8
# ==========================================
# 1. 패키지 임포트
# ==========================================

import os
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
from mmengine.config import Config
from mmengine.runner import Runner
from mmengine.model import BaseModule
from mmseg.registry import TRANSFORMS, MODELS
from mmcv.transforms import BaseTransform

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


# ==========================================
# 1-1. 시드 고정 (재현성 보장)
# ==========================================

def set_seed(seed=42):
    """
    모든 랜덤 시드를 고정하여 재현 가능한 결과 생성
    
    재현성이 보장되는 요소:
    - Python random
    - NumPy random
    - PyTorch CPU/GPU random
    - 데이터 샘플링 순서
    
    Args:
        seed (int): 시드 값 (기본값: 42)
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # ⭐ cudnn.deterministic은 False로 (성능상 이유)
    # 시드는 여전히 고정되어 있어서 충분한 재현성 보장
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
    
    # Python 해시 시드
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    print(f"✅ Random seed fixed to {seed}")

# 시드 고정 실행
SEED = 42
set_seed(SEED)

PyTorch: 2.0.0a0+1767026
CUDA available: True
✅ Random seed fixed to 42


In [5]:
# ==========================================
# 2. 커스텀 Transform: 라벨 리매핑
# ==========================================

@TRANSFORMS.register_module()
class RemapLabels(BaseTransform):
    """
    학습 시에만 라벨 값을 동적으로 변환하는 Transform
    
    배경:
    - 원본 데이터: 산업단지(10), 배경(90)
    - 학습용 데이터: 산업단지(1), 배경(0)
    
    장점:
    - 원본 파일 수정 없음 (대회 규정 준수)
    - 메모리에서만 변환 수행
    - 표준 라벨 인덱스로 안정적 학습
    
    Args:
        original_values (tuple): 원본 라벨 값 (10, 90)
        target_values (tuple): 변환할 값 (1, 0)
    """
    
    def __init__(self, original_values=(10, 90), target_values=(1, 0)):
        self.original_values = original_values
        self.target_values = target_values
    
    def transform(self, results):
        """
        라벨 변환 수행
        
        Args:
            results (dict): 데이터 딕셔너리 (gt_seg_map 포함)
            
        Returns:
            dict: 변환된 라벨이 포함된 딕셔너리
        """
        if 'gt_seg_map' in results:
            seg_map = results['gt_seg_map']
            new_seg_map = seg_map.copy()
            
            # 값 변환: 10 -> 1, 90 -> 0
            for orig_val, target_val in zip(self.original_values, self.target_values):
                new_seg_map[seg_map == orig_val] = target_val
            
            results['gt_seg_map'] = new_seg_map
        
        return results

In [6]:
# ==========================================
# 3. 커스텀 Backbone: 1x1 Conv Adapter
# ==========================================

@MODELS.register_module()
class BackboneWithInputConv(BaseModule):
    """
    4채널 입력을 3채널로 변환하여 Pretrained Backbone 활용
    
    아키텍처:
        Input (4ch) → 1x1 Conv → Output (3ch) → Pretrained Backbone
        
    초기화 전략:
        RGB 채널: Identity mapping (1.0)
        NIR 채널: Zero initialization (학습으로 최적화)
    
    Args:
        backbone (dict): Pretrained backbone config
        in_channels (int): 입력 채널 수 (기본값: 4)
        out_channels (int): 출력 채널 수 (기본값: 3)
        bias (bool): 1x1 Conv에 bias 사용 여부 (기본값: False)
        init_cfg (dict, optional): 초기화 설정
    """

    def __init__(self,
                 backbone: dict,
                 in_channels: int = 4,
                 out_channels: int = 3,
                 bias: bool = False,
                 init_cfg=None):
        super().__init__(init_cfg)

        # Pretrained backbone은 3채널 입력 가정
        assert out_channels == 3, "Pretrained backbone expects 3 channels"
        
        # 1x1 Conv Adapter 생성
        self.adapter = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=bias)
        
        # Pretrained Backbone 생성
        self.backbone = MODELS.build(backbone)

        # RGB Identity 초기화
        with torch.no_grad():
            w = self.adapter.weight  # shape: [3, 4, 1, 1]
            w.zero_()  # 모든 가중치를 0으로 초기화
            
            # RGB 채널은 identity mapping
            w[0, 0, 0, 0] = 1.0  # R → R
            w[1, 1, 0, 0] = 1.0  # G → G
            w[2, 2, 0, 0] = 1.0  # B → B
            # w[:, 3, 0, 0] = 0.0  # NIR → 0 (이미 zero_()로 설정됨)
            
            if self.adapter.bias is not None:
                self.adapter.bias.zero_()
        
        print("✅ Adapter initialized: RGB identity, NIR zero")

    def forward(self, x):
        """
        Forward pass
        
        Args:
            x (Tensor): 입력 이미지 [B, 4, H, W]
            
        Returns:
            Tensor or tuple: Backbone 출력 (multi-scale features)
        """
        x = self.adapter(x)        # 4채널 → 3채널 변환
        return self.backbone(x)    # Pretrained backbone forward

In [5]:
# ==========================================
# 4. Config 설정
# ==========================================

# 데이터 경로 설정
data_root = 'raw_data/industrial_complex/'
work_dir = 'work_dirs/segformer_b5_adapter'
os.makedirs(work_dir, exist_ok=True)

# Config 딕셔너리 생성
cfg = Config(dict(
    # ----------------------------------
    # 기본 설정
    # ----------------------------------
    default_scope='mmseg',
    backend_args=None,
    work_dir=work_dir,
    
    # ----------------------------------
    # 데이터 전처리 (모델 내부)
    # ----------------------------------
    data_preprocessor=dict(
        type='SegDataPreProcessor',
        bgr_to_rgb=False,         # TIFF는 RGB 순서
        pad_val=0,                # 이미지 패딩 값
        seg_pad_val=255,          # 마스크 패딩 값 (ignore index)
        size=None,                # 리사이즈 안 함
        size_divisor=32           # 32의 배수로 패딩
    ),
    
    # ----------------------------------
    # 모델 구조
    # ----------------------------------
    model=dict(
        type='EncoderDecoder',
        data_preprocessor=dict(
            type='SegDataPreProcessor',
            bgr_to_rgb=False,
            pad_val=0,
            seg_pad_val=255,
            size=None,
            size_divisor=32
        ),
        
        # Backbone: 1x1 Conv Adapter + Pretrained MiT-B5
        backbone=dict(
            type='BackboneWithInputConv',
            in_channels=4,              # 4채널 입력 (RGBN)
            out_channels=3,             # 3채널 출력 (RGB)
            bias=False,                 # Bias 미사용
            backbone=dict(
                type='MixVisionTransformer',
                in_channels=3,          # Pretrained는 3채널
                embed_dims=64,          # MiT-B5
                num_stages=4,
                num_layers=[3, 6, 40, 3],  # MiT-B5 구조
                num_heads=[1, 2, 5, 8],
                patch_sizes=[7, 3, 3, 3],
                sr_ratios=[8, 4, 2, 1],
                out_indices=(0, 1, 2, 3),
                mlp_ratio=4,
                qkv_bias=True,
                drop_rate=0.0,
                attn_drop_rate=0.0,
                drop_path_rate=0.1,
                # ImageNet Pretrained Weight
                init_cfg=dict(
                    type='Pretrained',
                    checkpoint='https://download.openmmlab.com/mmsegmentation/v0.5/pretrain/segformer/mit_b5_20220624-658746d9.pth'
                )
            )
        ),
        
        # Decode Head: SegFormer Head
        decode_head=dict(
            type='SegformerHead',
            in_channels=[64, 128, 320, 512],  # MiT-B5 출력 채널
            in_index=[0, 1, 2, 3],            # 4개 stage 모두 사용
            channels=256,                      # 중간 채널 수
            dropout_ratio=0.1,                 # Dropout 비율
            num_classes=1,                     # Binary segmentation
            threshold=0.5,                     # 예측 임계값
            norm_cfg=dict(type='SyncBN', requires_grad=True),
            align_corners=False,
            # Loss 함수
            loss_decode=[
                dict(
                    type='CrossEntropyLoss',
                    use_sigmoid=True,      # Binary라서 sigmoid 사용
                    loss_weight=1.0,
                )
            ]
        ),
        
        train_cfg=dict(),
        test_cfg=dict(mode='whole'),  # 전체 이미지 추론
    ),
    
    # ----------------------------------
    # 데이터 파이프라인
    # ----------------------------------
    
    # Train Pipeline (Augmentation 포함)
    train_pipeline=[
        dict(type='LoadImageFromFile', backend_args=None, imdecode_backend='tifffile'),
        dict(type='LoadAnnotations', backend_args=None, imdecode_backend='tifffile'),
        # 라벨 리매핑: 10→1, 90→0
        dict(type='RemapLabels', original_values=(10, 90), target_values=(1, 0)),
        # 정규화 (데이터셋 통계값)
        dict(
            type='Normalize',
            mean=[2110.71, 2048.15, 1828.06, 3291.53],
            std=[633.93, 523.48, 533.74, 859.42]
        ),
        dict(type='PackSegInputs'),
    ],
    
    # Validation Pipeline (Augmentation 없음)
    val_pipeline=[
        dict(type='LoadImageFromFile', backend_args=None, imdecode_backend='tifffile'),
        dict(type='LoadAnnotations', backend_args=None, imdecode_backend='tifffile'),
        dict(type='RemapLabels', original_values=(10, 90), target_values=(1, 0)),
        dict(
            type='Normalize',
            mean=[2110.71, 2048.15, 1828.06, 3291.53],
            std=[633.93, 523.48, 533.74, 859.42]
        ),
        dict(type='PackSegInputs'),
    ],
    
    # Test Pipeline (Augmentation 없음)
    test_pipeline=[
        dict(type='LoadImageFromFile', backend_args=None, imdecode_backend='tifffile'),
        dict(type='LoadAnnotations', backend_args=None, imdecode_backend='tifffile'),
        dict(type='RemapLabels', original_values=(10, 90), target_values=(1, 0)),
        dict(
            type='Normalize',
            mean=[2110.71, 2048.15, 1828.06, 3291.53],
            std=[633.93, 523.48, 533.74, 859.42]
        ),
        dict(type='PackSegInputs'),
    ],
    
    # ----------------------------------
    # 데이터로더
    # ----------------------------------
    
    # 메타 정보
    metainfo=dict(
        classes=('industrial_complex',),
        palette=[[0, 255, 0]]  # 시각화용 색상
    ),
    
    # Train Dataloader
    train_dataloader=dict(
        batch_size=8,
        num_workers=12,
        persistent_workers=True,
        sampler=dict(type='InfiniteSampler', shuffle=True),
        dataset=dict(
            type='BaseSegDataset',
            data_root=data_root,
            metainfo=dict(classes=('industrial_complex',), palette=[[0, 255, 0]]),
            data_prefix=dict(img_path='img_dir/train', seg_map_path='ann_dir/train'),
            img_suffix='.tif',
            seg_map_suffix='.tif',
            pipeline=None,  # train_pipeline 연결됨
        ),
    ),
    
    # Validation Dataloader
    val_dataloader=dict(
        batch_size=8,
        num_workers=4,
        persistent_workers=True,
        sampler=dict(type='DefaultSampler', shuffle=False),
        dataset=dict(
            type='BaseSegDataset',
            data_root=data_root,
            metainfo=dict(classes=('industrial_complex',), palette=[[0, 255, 0]]),
            data_prefix=dict(img_path='img_dir/val', seg_map_path='ann_dir/val'),
            img_suffix='.tif',
            seg_map_suffix='.tif',
            pipeline=None,  # val_pipeline 연결됨
        ),
    ),
    
    # Test Dataloader
    test_dataloader=dict(
        batch_size=8,
        num_workers=4,
        persistent_workers=True,
        sampler=dict(type='DefaultSampler', shuffle=False),
        dataset=dict(
            type='BaseSegDataset',
            data_root=data_root,
            metainfo=dict(classes=('industrial_complex',), palette=[[0, 255, 0]]),
            data_prefix=dict(img_path='img_dir/val', seg_map_path='ann_dir/val'),
            img_suffix='.tif',
            seg_map_suffix='.tif',
            pipeline=None,  # test_pipeline 연결됨
        ),
    ),
    
    # ----------------------------------
    # 평가 지표
    # ----------------------------------
    val_evaluator=dict(type='IoUMetric', iou_metrics=['mIoU']),
    test_evaluator=dict(type='IoUMetric', iou_metrics=['mIoU']),
    
    # ----------------------------------
    # 옵티마이저
    # ----------------------------------
    optim_wrapper=dict(
        type='OptimWrapper',
        optimizer=dict(
            type='AdamW',
            lr=6e-5,                    # Learning rate
            betas=(0.9, 0.999),         # Adam beta 파라미터
            weight_decay=0.01           # Weight decay
        )
    ),
    
    # ----------------------------------
    # 학습률 스케줄러 (Warmup + PolyLR)
    # ----------------------------------
    param_scheduler=[
        # Warmup: 처음 5000 iter 동안 lr을 0.001배에서 1배로 증가
        dict(
            type='LinearLR',
            start_factor=0.001,         # 시작 lr = 6e-5 * 0.001 = 6e-8
            by_epoch=False,
            begin=0,
            end=5000                    # 5000 iter까지 warmup
        ),
        # Main Scheduler: 5000 iter부터 320000 iter까지 PolyLR
        dict(
            type='PolyLR',              # Polynomial learning rate decay
            begin=5000,
            end=320000,                 # ⭐ 320,000 iter로 변경
            eta_min=1e-7,               # 최소 learning rate
            power=0.9,                  # Decay power
            by_epoch=False              # Iteration 기반
        )
    ],
    
    # ----------------------------------
    # 학습 설정
    # ----------------------------------
    train_cfg=dict(
        type='IterBasedTrainLoop',
        max_iters=320000,              # ⭐ 320,000 iter로 변경
        val_interval=10000             # ⭐ 10,000 iter마다 검증
    ),
    val_cfg=dict(type='ValLoop'),
    test_cfg=dict(type='TestLoop'),
    
    # ----------------------------------
    # 훅 (Hooks)
    # ----------------------------------
    default_hooks=dict(
        # 로그 출력
        logger=dict(
            type='LoggerHook',
            interval=100,               # ⭐ 100 iter마다 로그 출력
            log_metric_by_epoch=False
        ),
        # Learning rate 업데이트
        param_scheduler=dict(type='ParamSchedulerHook'),
        # 시간 측정
        timer=dict(type='IterTimerHook'),
        # 샘플러 시드
        sampler_seed=dict(type='DistSamplerSeedHook'),
        # 시각화
        visualization=dict(type='SegVisualizationHook', draw=False),
        # 체크포인트 저장
        checkpoint=dict(
            type='CheckpointHook',
            by_epoch=False,
            interval=10000,             # 10000 iter마다 저장
            max_keep_ckpts=3,           # 최대 3개 보관
            save_best='mIoU'            # mIoU 기준 best 모델 저장
        )
    ),
    
    # ----------------------------------
    # 시각화 백엔드
    # ----------------------------------
    vis_backends=[
        dict(type='LocalVisBackend', save_dir=work_dir)
    ],
    
    visualizer=dict(
        type='SegLocalVisualizer',
        vis_backends=[
            dict(type='LocalVisBackend', save_dir=work_dir)
        ],
        name='visualizer'
    ),
    
    # ----------------------------------
    # 환경 설정
    # ----------------------------------
    env_cfg=dict(
        cudnn_benchmark=True,          # 재현성을 위해 False
        mp_cfg=dict(
            mp_start_method='fork',
            opencv_num_threads=0
        ),
        dist_cfg=dict(backend='nccl')   # 분산 학습 백엔드
    ),
    
    # ----------------------------------
    # 랜덤 시드 설정
    # ----------------------------------
    randomness=dict(
        seed=SEED,                      # 시드 값
        deterministic=False,             
        diff_rank_seed=False            # 모든 rank 동일 시드
    ),
    
    launcher='none',                    # 'none', 'pytorch', 'slurm' 등
    log_level='INFO',
))

# Pipeline 연결
cfg.train_dataloader.dataset.pipeline = cfg.train_pipeline
cfg.val_dataloader.dataset.pipeline = cfg.val_pipeline
cfg.test_dataloader.dataset.pipeline = cfg.test_pipeline

print("✅ Config 설정 완료\n")

✅ Config 설정 완료



In [6]:
# ==========================================
# 5. 학습 실행
# ==========================================

print("=" * 60)
print("학습 시작")
print("=" * 60)
print(f"Model: SegFormer MiT-B5")
print(f"Input: 4-channel (RGB + NIR)")
print(f"Strategy: 1x1 Conv Adapter with RGB Identity Initialization")
print(f"Max Iterations: {cfg.train_cfg.max_iters}")
print(f"Validation Interval: {cfg.train_cfg.val_interval}")
print(f"Random Seed: {SEED}")
print("=" * 60)
print()

# Runner 생성 및 학습 실행
runner = Runner.from_cfg(cfg)
runner.train()

print("\n✅ 학습 완료!")

학습 시작
Model: SegFormer MiT-B5
Input: 4-channel (RGB + NIR)
Strategy: 1x1 Conv Adapter with RGB Identity Initialization
Max Iterations: 320000
Validation Interval: 10000
Random Seed: 42

10/13 20:23:03 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.8.10 (default, Nov 14 2022, 12:59:47) [GCC 9.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 42
    GPU 0: NVIDIA H100 80GB HBM3
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.1, V12.1.66
    GCC: x86_64-linux-gnu-gcc (Ubuntu 9.4.0-1ubuntu1~20.04.1) 9.4.0
    PyTorch: 2.0.0a0+1767026
    PyTorch compiling details: PyTorch built with:
  - GCC 9.4
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2021.1-Product Build 20201104 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.7.3 (Git Hash N/A)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usu

In [7]:
import os
import torch
from mmengine.config import Config
from mmengine.runner import Runner

# 경로 설정
work_dir = 'work_dirs/segformer_b5_adapter'
checkpoint_path = os.path.join(work_dir, 'best_mIoU_iter_310000.pth')
config_path = os.path.join(work_dir, 'config.py')

# Config 로드
cfg = Config.fromfile(config_path)
cfg.load_from = checkpoint_path

# Runner 생성 및 평가
print("=" * 60)
print(f"Evaluating: {checkpoint_path}")
print("=" * 60)

runner = Runner.from_cfg(cfg)
test_metrics = runner.test()

# 결과 출력
print("\n" + "=" * 60)
print("Test Results")
print("=" * 60)
for key, value in test_metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")
print("=" * 60)

Evaluating: work_dirs/segformer_b5_adapter/best_mIoU_iter_310000.pth
10/14 22:45:30 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.8.10 (default, Nov 14 2022, 12:59:47) [GCC 9.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 42
    GPU 0: NVIDIA H100 80GB HBM3
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.1, V12.1.66
    GCC: x86_64-linux-gnu-gcc (Ubuntu 9.4.0-1ubuntu1~20.04.1) 9.4.0
    PyTorch: 2.0.0a0+1767026
    PyTorch compiling details: PyTorch built with:
  - GCC 9.4
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2021.1-Product Build 20201104 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.7.3 (Git Hash N/A)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX2
  - CUDA Runtime 12.1
  - NVCC architectur